# Student Performance Factors: Data Cleaning & Descriptive Statistics

G0251 Foundations in Business Analytics — Descriptive Statistics and Predictive Analysis

This notebook covers the data cleaning and descriptive-statistics stages of the group project. It reproduces the cleaning steps, calculated column, and summary statistics reported in `docs/report/Descriptive_Statistics_and_Predictive_Analysis.docx`.

Dataset: [Student Performance Factors](https://www.kaggle.com/datasets/ayeshasiddiqa123/student-performance) (Kaggle).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew

df_raw = pd.read_csv("../data/StudentPerformanceFactors.csv")
print("Shape:", df_raw.shape)
df_raw.head()

## 1. Data Cleaning and Preparation

The original dataset has **6,607 rows and 20 columns**. Initial inspection revealed 235 missing values across 3 columns, 1,463 invalid outlier-type values, and 1 exact duplicate row.

### 1.1 Column Selection

Nine columns were dropped as not meaningfully predictive of exam performance, or too imbalanced / high-missingness to be useful:

In [ ]:
columns_to_drop = [
    "Parental_Involvement",     # not something the student can directly control
    "Sleep_Hours",               # indirect link to academic performance
    "Motivation_Level",          # self-reported, subjective
    "Internet_Access",           # 92.4% "Yes" -- doesn't help distinguish outcomes
    "Peer_Influence",            # external social factor, beyond core academics
    "Physical_Activity",         # lifestyle variable, indirect influence
    "Learning_Disabilities",     # 89.5% "No", highly imbalanced
    "Parental_Education_Level",  # parental attribute + missing values
    "Distance_from_Home",        # weak academic link + missing values
]

df = df_raw.drop(columns=[c for c in columns_to_drop if c in df_raw.columns])
print("Shape after column selection:", df.shape)

### 1.2 Replacing Missing Values

After column selection, the only remaining missing values are in `Teacher_Quality` (78 missing, ~1.18% of records).

In [ ]:
print("Missing values after column selection:")
print(df.isna().sum()[df.isna().sum() > 0])

# Teacher_Quality is categorical -- fill with mode
if "Teacher_Quality" in df.columns:
    df["Teacher_Quality"] = df["Teacher_Quality"].fillna(df["Teacher_Quality"].mode()[0])

print("\nMissing values after imputation:", df.isna().sum().sum())

### 1.3 Outlier / Invalid Value Handling

Interval validation rules were applied to detect logically impossible values:

| Column | Valid Range | Invalid Count | Invalid Values | Replacement |
|---|---|---|---|---|
| Hours_Studied | [0, 24] | 1,462 | 25-44 | Median (20.0) |
| Exam_Score | [0, 100] | 1 | 101 | Median (67.0) |
| Attendance | [0, 100] | 0 | -- | None needed |
| Previous_Scores | [0, 100] | 0 | -- | None needed |

In [ ]:
# Hours_Studied cannot exceed 24 in a day
invalid_hours = df["Hours_Studied"] > 24
print("Invalid Hours_Studied records:", invalid_hours.sum())
df.loc[invalid_hours, "Hours_Studied"] = df["Hours_Studied"].median()

# Exam_Score cannot exceed 100
invalid_scores = df["Exam_Score"] > 100
print("Invalid Exam_Score records:", invalid_scores.sum())
df.loc[invalid_scores, "Exam_Score"] = df["Exam_Score"].median()

### 1.4 Duplicate Data Handling

In [ ]:
dup_count = df.duplicated().sum()
print("Exact duplicate rows found:", dup_count)
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

### 1.5 New Column: Score_Improvement

A calculated column comparing exam performance to prior academic standing:

`Score_Improvement = Exam_Score - Previous_Scores`

Positive values indicate improvement; negative values indicate decline.

In [ ]:
df["Score_Improvement"] = df["Exam_Score"] - df["Previous_Scores"]
df[["Exam_Score", "Previous_Scores", "Score_Improvement"]].head()

### 1.6 Cleaning Summary

| Attribute | Before Cleaning | After Cleaning |
|---|---|---|
| Total Rows | 6,607 | 6,606 |
| Total Columns | 20 | 12 (11 + 1 calculated) |
| Missing Values | 235 (across 3 columns) | 0 |
| Invalid Hours (>24) | 1,462 records | 0 (replaced with median) |
| Invalid Exam Score (>100) | 1 record | 0 (replaced with median) |
| Duplicate Rows | 1 | 0 (removed) |
| Derived Features | None | Score_Improvement |

In [ ]:
print("Final cleaned dataset:")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 2. Descriptive Analysis

Statistical measures for the six numerical variables: central tendency (mean, median, mode), dispersion (standard deviation, variance, IQR), and shape (skewness).

In [ ]:
numeric_vars = [
    "Hours_Studied", "Attendance", "Previous_Scores",
    "Tutoring_Sessions", "Exam_Score", "Score_Improvement"
]

summary_rows = []
for col in numeric_vars:
    s = df[col]
    summary_rows.append({
        "Variable": col,
        "Mean": round(s.mean(), 2),
        "Std Deviation": round(s.std(), 2),
        "25th Percentile": s.quantile(0.25),
        "Median": s.median(),
        "75th Percentile": s.quantile(0.75),
        "Mode": s.mode()[0],
        "Variance": round(s.var(), 2),
        "Skewness": round(skew(s.dropna()), 4),
        "IQR": s.quantile(0.75) - s.quantile(0.25),
    })

descriptive_stats = pd.DataFrame(summary_rows).set_index("Variable")
descriptive_stats

### Interpretation

- **Exam Score**: mean 67.23, median 67.00, mode 67.00 -- close together, suggesting a fairly symmetric core distribution, though **highly right-skewed (2.13)**, meaning a small number of students scored well above the typical range.
- **Hours Studied**: mean 17.76 hours/week, median 18.00; **moderately left-skewed (-0.58)** -- most students study relatively more hours, with only a few studying significantly less.
- **Attendance**: mean 79.98%, median 80.00% -- typical students attend roughly 4 out of 5 classes.
- **Previous Scores vs Exam Score**: the gap between Previous Scores (mean 75.07) and Exam Score (mean 67.23) is the most visible signal in these statistics -- students on average score lower on the final exam than their historical average, consistent with the negative mean Score Improvement (-7.84).
- **Tutoring Sessions**: mean only 1.49, median 1.00 -- most students receive very little outside academic support.
- Attendance, Previous Scores, and Score Improvement all show near-zero skewness (approximately symmetric distributions); Tutoring Sessions shows a slight right skew (0.35).

## 3. Visualizations

### 3.1 Correlation Matrix Heatmap

See `docs/images/correlation_matrix.png` for the rendered chart from the original report.

In [ ]:
corr = df[numeric_vars].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdYlBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels([c.replace("_", " ") for c in corr.columns], rotation=45, ha="right")
ax.set_yticklabels([c.replace("_", " ") for c in corr.columns])
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Correlation Matrix of Numerical Variables")
fig.colorbar(im)
plt.tight_layout()
plt.show()

**Insight:** Attendance shows the strongest correlation with Exam Score (r ≈ 0.58), followed by Hours Studied (r ≈ 0.29). Both are weak-to-moderate positive relationships -- consistent, but not dominant, predictors on their own. Note the r ≈ -0.96 between Previous Scores and Score Improvement is expected and largely mechanical, since `Score_Improvement = Exam_Score - Previous_Scores` by construction.

### 3.2 Hours Studied vs Exam Score

See `docs/images/hours_vs_exam_score.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["Hours_Studied"], df["Exam_Score"], alpha=0.15, s=15)

z = np.polyfit(df["Hours_Studied"], df["Exam_Score"], 1)
trend = np.poly1d(z)
xs = np.linspace(df["Hours_Studied"].min(), df["Hours_Studied"].max(), 100)
ax.plot(xs, trend(xs), color="red", label=f"Trend (slope={z[0]:.2f})")

ax.set_title("Hours Studied vs Exam Score")
ax.set_xlabel("Hours Studied")
ax.set_ylabel("Exam Score")
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** The trend line has a moderate positive slope, consistent with the weak positive correlation from the heatmap -- more study hours are associated with modestly higher exam scores, but study time alone explains only part of the variation.

### 3.3 Attendance vs Exam Score by Gender

See `docs/images/attendance_vs_exam_by_gender.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for gender, color in [("Male", "tab:blue"), ("Female", "tab:orange")]:
    subset = df[df["Gender"] == gender]
    ax.scatter(subset["Attendance"], subset["Exam_Score"], alpha=0.2, s=15, label=gender, color=color)

ax.set_title("Attendance vs Exam Score by Gender")
ax.set_xlabel("Attendance (%)")
ax.set_ylabel("Exam Score")
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** Both genders show a similarly weak positive relationship between attendance and exam performance -- neither group shows a visibly different pattern, suggesting gender is not a meaningful moderator of the attendance-performance relationship in this dataset.

## 4. Predictive Analysis Framing

This section documents *potential* predictive modeling directions identified for the dataset; no model was trained as part of this notebook (see `docs/report/` for the full write-up including the Power BI dashboard).

**1. Exam Score Prediction (Regression)** -- target: `Exam_Score` (continuous). Predictors: Hours Studied, Attendance, Previous Scores, Tutoring Sessions, Teacher Quality, Access to Resources, Family Income, School Type.

**2. Risk Status Classification (Classification)** -- target: binary `Risk_Status` (Exam Score < 60 = at-risk). Predictors: same feature set as above. Suggested models: Decision Trees, Logistic Regression, for interpretability.

**Business value:** early identification of at-risk students, optimized allocation of tutoring/counseling resources, and data-driven policy decisions for academic administrators.